# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² tabular dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object (do not subscript/iterate over it)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets (each with its `@id`, name, and available fields)
print("Record sets in this dataset:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    print(f"  name: {rs.get('name', None)}")
    # Print available fields by @id
    fields = rs.get('field', [])
    if isinstance(fields, dict):  # A single field
        fields = [fields]
    elif not isinstance(fields, list):  # None or unexpected
        fields = []
    if fields:
        print("  fields:")
        for field in fields:
            # In mlcroissant schema, each field is a dict with '@id' and 'name'
            print(f"    - @id: {field['@id']}, name: {field.get('name', None)}")
    print()
if not record_sets:
    print("No record sets found in the metadata.")
# For demo purposes, get the first available record set for extraction later
first_rs_id = None
if record_sets:
    first_rs_id = record_sets[0]['@id']

## 3. Data Extraction

Load data from the main tabular record set into a DataFrame for analysis. All extraction references record sets and fields by their precise `@id`.

In [ ]:
# Get all record set @ids for extraction
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded record set: {rs_id} with shape: {df.shape}")
        else:
            print(f"Record set {rs_id}: No records available.")
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}")

# Pick the main clinical table for demonstration (choose the first one if available)
main_rs_id = first_rs_id

if main_rs_id in dataframes:
    print(f"\nColumns in {main_rs_id}:\n", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No main table found to display.")

## 4. Exploratory Data Analysis (EDA)

Basic field statistics and grouping (referencing all fields by their `@id`).

In [ ]:
# Identify a numeric field and a group field using @id (replace these with actual field @ids as found above)
numeric_field_id = None
group_field_id = None

# Try to auto-detect candidate field names
if main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    for col in df.columns:
        # Try to find a column with likely numeric values
        if numeric_field_id is None and pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
        # Try to find a grouping field (e.g. gender, location, diagnosis)
        if group_field_id is None and pd.api.types.is_object_dtype(df[col]) and col.lower().find('sex') != -1:
            group_field_id = col

# Report what we chose
print(f"Selected numeric field: {numeric_field_id}")
print(f"Selected group field: {group_field_id}")

if numeric_field_id and numeric_field_id in df.columns:
    # Use an arbitrary threshold for filtering
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print("No appropriate numeric field found for analysis.")

## 5. Visualization

Visualize the distribution of the numeric field and its breakdown by group, referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Cannot plot distributions: numeric field or data missing.")

## 6. Conclusion

In this notebook, we've demonstrated how to:
- Load the FAIR² colorectal cancer survivors dataset via its Croissant schema using `mlcroissant`
- Inspect record sets and fields using `@id` references
- Extract tabular data into Pandas DataFrames
- Perform simple EDA: filter records, normalize numeric fields, and compare distributions across groups
- Visualize key metrics for further clinical studies.

This approach generalizes to any Croissant-compatible dataset and allows for reproducible, transparent biomedical data science workflows.

*Remember: All specific record sets, fields, and columns were referenced by their `@id` throughout this workflow.*